# Phase 8: Classifier Model Export to TorchScript

This notebook traces the trained EfficientNet-B0 PyTorch classifier using `torch.jit.trace` and compiles it into a **TorchScript (`model.ts`)** binary. This binary is loaded directly by the FastAPI backend server for low-latency REST API prediction execution.

In [1]:
# Setup paths and imports
from pathlib import Path
import sys
import torch
import torch.nn as nn
import json
import time

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'ai' and PROJECT_ROOT.parent != PROJECT_ROOT:
    if (PROJECT_ROOT / 'ai').exists():
        PROJECT_ROOT = PROJECT_ROOT / 'ai'
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import utils

In [2]:
# Load accelerator and best PyTorch state dict
device = utils.device.get_device()

def build_classifier(num_classes):
    import torchvision.models as models
    model = models.efficientnet_b0()
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

model = build_classifier(utils.config.NUM_CLASSES).to(device)

if utils.config.BEST_MODEL_PATH.exists():
    checkpoint = torch.load(str(utils.config.BEST_MODEL_PATH), map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print("✅ Loaded model weights successfully.")
else:
    print("⚠️ Warning: Best weights checkpoint not found. Using random weights.")

✅ Loaded model weights successfully.


In [3]:
# Trace PyTorch model using a dummy input tensor
dummy_input = torch.randn(1, 3, utils.config.IMAGE_SIZE, utils.config.IMAGE_SIZE).to(device)
torchscript_model = torch.jit.trace(model, dummy_input)

# Export path to production models/classifier/ folder
export_path = utils.paths.CLASSIFIER_MODEL_DIR / "model.ts"
export_path.parent.mkdir(parents=True, exist_ok=True)

torchscript_model.save(str(export_path))
print("✅ TorchScript model compiled and exported successfully to:", export_path)

✅ TorchScript model compiled and exported successfully to: O:\Hackthons\KrishiOS\ai\models\classifier\model.ts


In [4]:
# Verify load validity of the exported binary
loaded_model = torch.jit.load(str(export_path), map_location=device)
loaded_model.eval()
print("✅ Loaded exported TorchScript model successfully.")

# Verify mathematical output parity
with torch.no_grad():
    original_output = model(dummy_input)
    exported_output = loaded_model(dummy_input)

diff = torch.max(torch.abs(original_output - exported_output))
print(f"Maximum output discrepancy: {diff.item():.8f}")
assert diff.item() < 1e-5, "Verification failed: Discrepancy is too high!"

✅ Loaded exported TorchScript model successfully.
Maximum output discrepancy: 0.00000000
